# Import library

In [ ]:
import os
import sys
import time
import datetime

from termcolor import colored, cprint

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
from os import listdir

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations
from abyss.helpers import metrics_viewer
from abyss.models import create_decoder
from abyss.glob import glob
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
from sklearn.metrics import roc_curve, auc

phenotypes_labels = {
    "bc": "Breast Cancer",
    "cad": "Coronary Artery Disease",
    "ms": "Multiple Sclerosis",
    "t2d": "Type 2 Diabetes",
    "pc": "Prostate Cancer",
    "ibd": "Inflammatory Bowel Disease"
}

scenario_labels = {
    "no_cov": "no covariates",
    "abyss": "Abyss correction",
    "cov": "PCA correction",
    "abyss_cov": "Abyss + PCA as covariates"
}

In [ ]:
def ld_score_regression_plot(ld_scores, chi_squared_stats, weights):
    # Existing plot code
    specified_weights = [0.2, 0.4, 0.6, 0.8, 1.0]
    cmap = sns.color_palette("rocket", as_cmap=True)
    norm = plt.Normalize(min(weights), max(weights))

    # Perform weighted linear regression
    X = sm.add_constant(ld_scores)
    model = sm.WLS(chi_squared_stats, X, weights=weights).fit()

    # Extract the slope (beta) and intercept from the model
    slope = model.params.iloc[1]
    intercept = model.params.iloc[0]

    # Generate x-values for the regression line
    x_range = np.linspace(min(ld_scores), max(ld_scores), 100)

    # Calculate y-values for the regression line
    y_range = slope * x_range + intercept

    # Plot the regression line with a thinner line
    plt.plot(x_range, y_range, color='black', linewidth=1)  # Adjust linewidth here
    
    for weight in specified_weights:
        plt.scatter([], [], color=cmap(norm(weight)), alpha=0.5, label=f'{weight:.1f}')

    scatter = plt.scatter(ld_scores, chi_squared_stats, c=weights, cmap=cmap, s=10)
    legend = plt.legend(title='Regression weight', frameon=False, scatterpoints=1, loc='upper right')
    plt.xlabel('LD Score bin')
    plt.ylabel(r'Mean $\chi^2$')

    # Adjusting the legend to include the regression line label
    handles, labels = plt.gca().get_legend_handles_labels()
    new_legend = plt.legend(handles, labels, title='Regression weight', frameon=False, scatterpoints=1, loc='upper right')

    # Show the plot
    plt.show()
    return slope, intercept

In [ ]:
diseases_LD = []
scenarios_LD = []
bins_LD = []
beta_LD = []
intercept_LD = []

In [ ]:
disease = "cad"
scenario = "no_cov"
bin = "5"
nr_snps = 2000
PCs = 15
id_samples = pd.read_pickle(f"{glob.PATH_usefull}/id_samples.pkl")
pheno = pd.read_pickle(glob.PATH_PHENO + f"/{disease}_holdout.pkl")

In [ ]:
scenarios = ['cov', 'no_cov', 'abyss', 'abyss_cov']
diseases = ['ms','cad']

In [ ]:
for disease in diseases:
    for scenario in scenarios:
        path_input = glob.PATH_GENO + f"/09_summary_statistics/{disease}/{scenario}"
        list_of_bins = os.listdir(path_input)
        list_of_dfs = []
        for biny in list_of_bins[0:1]:
            biny = f"bin_{bin}_sumstat.pkl"
            try:
                temp = pd.read_pickle(path_input+f"/{biny}")
                temp['bin'] = biny.split('_')[1]
                
                # Sort by -logp in descending order and take the top 1000 SNPs
                top_snps = temp.sort_values(by='-logp', ascending=False).head(nr_snps)
                
                list_of_dfs.append(top_snps)
            except Exception as e:
                print(e)
                print(f"Problem with chrom {biny}")
        
        # Concatenating the top SNPs from each bin
        sumstat = pd.concat(list_of_dfs)
        
        # Sorting by chromosome if needed
        sumstat = sumstat.sort_values(by='#CHROM', ascending=True)
        
        running_pos = 0
        cumulative_pos = []
        for chrom, group_df in sumstat.groupby('#CHROM'):
            cumulative_pos.append(group_df['POS'] + running_pos)
            running_pos += group_df['POS'].max()
        
        sumstat['cumulative_pos'] = pd.concat(cumulative_pos)
        sumstat['SNP number'] = sumstat.index
        unique_bins = list(sumstat['bin'].unique())
    
        genos = []
        mafs = []
        
        # Iterate over unique bins
        for unique_bin in unique_bins:
            # Filter sumstat for the current bin
            temp_sumstat = sumstat[sumstat['bin'] == unique_bin]
        
            # Construct path for genotype mapping
            path_geno_mapping = os.path.join(glob.PATH_GENO, f"04_LD_blocks/{PCs}_PCs/bin_{unique_bin}")
            
            # Load mapping data
            map_file = os.path.join(path_geno_mapping, "mapping.pkl")
            map = pd.read_pickle(map_file)
            map.rename(columns={'Position': 'POS'}, inplace=True)
            map['POS'] = map['POS'].astype(int)
        
        
            # Merge sumstat with mapping data
            temp_sumstat = pd.merge(temp_sumstat, map[['POS','Chromosome','Alls2', 'path']], on='POS', how='inner')
            
            # Process each unique path
            unique_paths = temp_sumstat['path'].unique()
            for path in unique_paths:
                temp2_sumstat = temp_sumstat[temp_sumstat['path'] == path]
                SNPS = temp2_sumstat['snp'].tolist()
                file_with_snp = next(f for f in os.listdir(path_geno_mapping) if f.startswith(path))
                path_file_genos = os.path.join(path_geno_mapping, file_with_snp)
                geno = pd.read_pickle(path_file_genos)[SNPS]
                temp = geno.T
                temp['snp'] = list(temp.index)
                #temp = pd.merge(temp, sumstat[['-logp', 'cumulative_pos','snp']], on='snp', how='inner')
                temp = pd.merge(temp, sumstat[['beta_snp','snp']], on='snp', how='inner')
                
                temp = temp.set_index('snp')
                geno = temp.T
                genos.append(geno)
    
        combined_geno = pd.concat(genos, axis=1)
        combined_geno = combined_geno.iloc[:-1].copy()
        combined_geno['FID'] = list(id_samples['FID'])
        combined_geno = pd.merge(combined_geno, pheno[['FID']], on='FID')
        combined_geno = combined_geno.drop('FID', axis=1)
        r_squared_matrix = combined_geno.corr() ** 2
        
        # Sum r^2 values for each SNP to get LD scores
        ld_scores = r_squared_matrix.sum()
        
        ld_score_df = pd.DataFrame(list(ld_scores.items()), columns=['snp', 'LD_score'])
        merged_data = pd.merge(ld_score_df, sumstat, on='snp')
        merged_data['heritability'] = (merged_data['beta_snp'] ** 2) / (merged_data['std_err'] ** 2)
        
        merged_data['X2'] = (merged_data['beta_snp'] ** 2) / (merged_data['std_err'] ** 2)
        N = merged_data.shape[0]
        M = pheno.shape[0]
        
        # Assuming you have already sorted your dataframe by 'LD_score'
        merged_data_sorted = merged_data.sort_values(by='LD_score')
        
        # Define the number of bins (100 in your case)
        num_bins = 100
        
        # Create the bins for LD scores
        bins = pd.cut(merged_data_sorted['LD_score'], bins=num_bins)
        
        # Group the data by the LD score bins and calculate the mean LD_score and X2 for each bin
        mean_ld_x2 = merged_data_sorted.groupby(bins, observed=False).agg({'LD_score': 'mean', 'X2': 'mean'})
        
        # Create a new DataFrame with mean LD scores and X2 for each LD score bin
        new_df = pd.DataFrame({
            'LD_score': mean_ld_x2['LD_score'],
            'X2': mean_ld_x2['X2'],
            'LD_score_bin': mean_ld_x2.index.categories
        })
        
        # Sort the new DataFrame by 'LD_score_bin' if needed
        new_df = new_df.sort_values(by='LD_score_bin')
        
        # Reset the index for the new DataFrame if desired
        new_df = new_df.reset_index(drop=True)
        
        # Now, new_df contains mean LD scores and mean X2 for each LD score bin
        new_df['normalized_ld_score'] = (np.max(new_df['LD_score'])/new_df['LD_score'])/np.max((np.max(new_df['LD_score'])/new_df['LD_score']))
        new_df = new_df.dropna()
    
        slope, intercept = ld_score_regression_plot(new_df['LD_score'],new_df['X2'], weights = new_df['normalized_ld_score'])
        diseases_LD.append(disease)
        scenarios_LD.append(scenario)
        bins_LD.append(bin)
    
        beta_LD.append(slope)
        intercept_LD.append(intercept)

In [ ]:
ld_dataframe = pd.DataFrame({
    'Disease': diseases_LD,
    'Scenario': scenarios_LD,
    'Bin': bins_LD,
    'Beta': beta_LD,
    'Intercept': intercept_LD
})

In [ ]:
ld_dataframe

In [ ]:
# Create a bar plot
sns.barplot(x='Scenario', y='Intercept', data=ld_dataframe)

# Adding labels and title
plt.xlabel('Scenario')
plt.ylabel('Intercept')
plt.title('Intercept Values by Scenario')

# Show the plot
plt.show()